In [4]:
# Restaurant Inventory Forecasting System - Imports and Setup
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("📦 All packages imported successfully!")

📦 All packages imported successfully!


In [12]:
class RestaurantInventoryForecaster:
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.food_items = [
            'Chicken Breast', 'Beef Steak', 'Salmon', 'Pasta', 'Rice',
            'Tomatoes', 'Onions', 'Lettuce', 'Cheese', 'Bread',
            'Potatoes', 'Carrots', 'Bell Peppers', 'Mushrooms', 'Garlic'
        ]

    def generate_sample_data(self, days=365):
        """Generate realistic restaurant inventory data"""
        np.random.seed(42)

        data = []
        start_date = datetime.now() - timedelta(days=days)

        for i in range(days):
            date = start_date + timedelta(days=i)

            # Add seasonality and day-of-week effects
            day_of_week = date.weekday()  # 0=Monday, 6=Sunday
            month = date.month

            # Weekend multiplier
            weekend_mult = 1.3 if day_of_week >= 5 else 1.0

            # Seasonal multiplier
            if month in [12, 1, 2]:  # Winter
                seasonal_mult = 0.8
            elif month in [6, 7, 8]:  # Summer
                seasonal_mult = 1.2
            else:
                seasonal_mult = 1.0

            for item in self.food_items:
                # Base consumption with item-specific patterns
                base_consumption = {
                    'Chicken Breast': 25, 'Beef Steak': 15, 'Salmon': 12,
                    'Pasta': 30, 'Rice': 20, 'Tomatoes': 18, 'Onions': 15,
                    'Lettuce': 10, 'Cheese': 12, 'Bread': 35, 'Potatoes': 22,
                    'Carrots': 8, 'Bell Peppers': 10, 'Mushrooms': 7, 'Garlic': 3
                }[item]

                # Add randomness, seasonality, and day effects
                daily_consumption = max(0, int(
                    base_consumption *
                    weekend_mult *
                    seasonal_mult *
                    np.random.normal(1, 0.2)  # Random variation
                ))

                # Weather effect (random)
                weather_effect = np.random.choice(['sunny', 'rainy', 'cloudy'], p=[0.6, 0.2, 0.2])
                if weather_effect == 'rainy':
                    daily_consumption = int(daily_consumption * 0.8)
                elif weather_effect == 'sunny':
                    daily_consumption = int(daily_consumption * 1.1)

                # Special events (random)
                special_event = np.random.choice([0, 1], p=[0.9, 0.1])
                if special_event:
                    daily_consumption = int(daily_consumption * 1.5)

                data.append({
                    'date': date,
                    'food_item': item,
                    'day_of_week': day_of_week,
                    'month': month,
                    'is_weekend': int(day_of_week >= 5),
                    'weather': weather_effect,
                    'special_event': special_event,
                    'consumption': daily_consumption
                })

        return pd.DataFrame(data)

    def prepare_features(self, df):
      """Prepare features for machine learning"""
      # Create lag features
      df = df.sort_values(['food_item', 'date'])

      # Add rolling averages
      df['consumption_7day_avg'] = df.groupby('food_item')['consumption'].rolling(7).mean().values
      df['consumption_14day_avg'] = df.groupby('food_item')['consumption'].rolling(14).mean().values
      df['consumption_30day_avg'] = df.groupby('food_item')['consumption'].rolling(30).mean().values

      # Add lag features
      df['consumption_lag1'] = df.groupby('food_item')['consumption'].shift(1)
      df['consumption_lag7'] = df.groupby('food_item')['consumption'].shift(7)

      # Encode categorical variables
      df['weather_sunny'] = (df['weather'] == 'sunny').astype(int)
      df['weather_rainy'] = (df['weather'] == 'rainy').astype(int)

      # Fill NaN values - using modern pandas approach
      df = df.bfill().fillna(0)

      return df

    def train_models(self, df):
      """Train forecasting models for each food item"""
      # Store historical data for use in predictions
      self.historical_data = df.copy()
      
      df_prepared = self.prepare_features(df)

      feature_columns = [
          'day_of_week', 'month', 'is_weekend', 'special_event',
          'weather_sunny', 'weather_rainy',
          'consumption_7day_avg', 'consumption_14day_avg', 'consumption_30day_avg',
          'consumption_lag1', 'consumption_lag7'
      ]

      for item in self.food_items:
          item_data = df_prepared[df_prepared['food_item'] == item].copy()

          # Skip if not enough data
          if len(item_data) < 50:
              continue

          # Prepare training data
          X = item_data[feature_columns].values
          y = item_data['consumption'].values

          # Scale features
          scaler = StandardScaler()
          X_scaled = scaler.fit_transform(X)

          # Train model
          model = RandomForestRegressor(
              n_estimators=100,
              max_depth=10,
              random_state=42
          )
          model.fit(X_scaled, y)

          # Store model and scaler
          self.models[item] = model
          self.scalers[item] = scaler

      print(f"✅ Trained models for {len(self.models)} food items")

    def predict_consumption(self, food_item, date, weather='sunny', special_event=0):
        """Predict consumption for a specific item and date"""
        if food_item not in self.models:
            return None

        # Prepare features for prediction
        day_of_week = date.weekday()
        month = date.month
        is_weekend = int(day_of_week >= 5)
        weather_sunny = int(weather == 'sunny')
        weather_rainy = int(weather == 'rainy')

        # Get actual historical data for lag features and rolling averages
        if hasattr(self, 'historical_data') and self.historical_data is not None:
            # Filter historical data for this specific food item
            item_history = self.historical_data[
                self.historical_data['food_item'] == food_item
            ].copy()
            
            if not item_history.empty:
                # Sort by date to ensure proper order
                item_history = item_history.sort_values('date')
                
                # Get the most recent data (last 30 days for rolling averages)
                recent_data = item_history.tail(30)
                
                # Calculate rolling averages from actual data
                consumption_7day_avg = recent_data['consumption'].tail(7).mean() if len(recent_data) >= 7 else recent_data['consumption'].mean()
                consumption_14day_avg = recent_data['consumption'].tail(14).mean() if len(recent_data) >= 14 else recent_data['consumption'].mean()
                consumption_30day_avg = recent_data['consumption'].mean()
                
                # Calculate lag features from actual data
                consumption_lag1 = recent_data['consumption'].iloc[-1] if len(recent_data) >= 1 else consumption_7day_avg
                consumption_lag7 = recent_data['consumption'].iloc[-7] if len(recent_data) >= 7 else consumption_7day_avg
            else:
                # Fallback to default values if no historical data
                consumption_7day_avg = consumption_14day_avg = consumption_30day_avg = 20
                consumption_lag1 = consumption_lag7 = 20
        else:
            # Fallback to default values if no historical data available
            consumption_7day_avg = consumption_14day_avg = consumption_30day_avg = 20
            consumption_lag1 = consumption_lag7 = 20

        features = np.array([[
            day_of_week, month, is_weekend, special_event,
            weather_sunny, weather_rainy,
            consumption_7day_avg, consumption_14day_avg, consumption_30day_avg,  # rolling averages
            consumption_lag1, consumption_lag7  # lag features
        ]])

        # Scale features
        features_scaled = self.scalers[food_item].transform(features)

        # Make prediction
        prediction = self.models[food_item].predict(features_scaled)[0]
        return max(0, int(prediction))

    def forecast_inventory_needs(self, days_ahead=7, current_stock=None, safety_margin=0.2):
        """Forecast inventory needs for the next few days"""
        if current_stock is None:
            # Default current stock levels
            current_stock = {item: np.random.randint(50, 200) for item in self.food_items}

        forecast_results = {}

        for item in self.food_items:
            if item not in self.models:
                continue

            daily_predictions = []
            current_date = datetime.now()

            for day in range(days_ahead):
                future_date = current_date + timedelta(days=day)

                # Predict consumption (assuming sunny weather, no special events)
                predicted_consumption = self.predict_consumption(
                    item, future_date, weather='sunny', special_event=0
                )

                if predicted_consumption is not None:
                    daily_predictions.append({
                        'date': future_date.strftime('%Y-%m-%d'),
                        'predicted_consumption': predicted_consumption
                    })

            if daily_predictions:
                total_predicted_consumption = sum([p['predicted_consumption'] for p in daily_predictions])
                current_stock_level = current_stock.get(item, 0)

                # Calculate if restock is needed
                stock_needed = total_predicted_consumption * (1 + safety_margin)
                restock_needed = max(0, stock_needed - current_stock_level)

                forecast_results[item] = {
                    'current_stock': current_stock_level,
                    'predicted_consumption': total_predicted_consumption,
                    'stock_needed': int(stock_needed),
                    'restock_needed': int(restock_needed),
                    'daily_predictions': daily_predictions,
                    'status': 'RESTOCK NEEDED' if restock_needed > 0 else 'SUFFICIENT STOCK'
                }

        return forecast_results

    def print_forecast_summary(self, forecast_results, days_ahead=7):
        """Print a formatted forecast summary"""
        print(f"\n🍽️  RESTAURANT INVENTORY FORECAST - NEXT {days_ahead} DAYS")
        print("=" * 60)

        urgent_items = []
        sufficient_items = []

        for item, data in forecast_results.items():
            if data['status'] == 'RESTOCK NEEDED':
                urgent_items.append((item, data))
            else:
                sufficient_items.append((item, data))

        # Print urgent items first
        if urgent_items:
            print("\n🚨 URGENT - RESTOCK NEEDED:")
            print("-" * 30)
            for item, data in urgent_items:
                print(f"{item:15} | Current: {data['current_stock']:3d} | "
                        f"Need: {data['predicted_consumption']:3d} | "
                        f"Order: {data['restock_needed']:3d}")

        if sufficient_items:
            print(f"\n✅ SUFFICIENT STOCK ({len(sufficient_items)} items):")
            print("-" * 30)
            for item, data in sufficient_items[:5]:  # Show top 5
                print(f"{item:15} | Current: {data['current_stock']:3d} | "
                        f"Need: {data['predicted_consumption']:3d}")

            if len(sufficient_items) > 5:
                print(f"... and {len(sufficient_items) - 5} more items with sufficient stock")

    def print_forecast_summary(self, forecast_results, days_ahead=7):
        """Print a formatted forecast summary"""
        print(f"\n🍽️  RESTAURANT INVENTORY FORECAST - NEXT {days_ahead} DAYS")
        print("=" * 60)

        urgent_items = []
        sufficient_items = []

        for item, data in forecast_results.items():
            if data['status'] == 'RESTOCK NEEDED':
                urgent_items.append((item, data))
            else:
                sufficient_items.append((item, data))

        # Print urgent items first
        if urgent_items:
            print("\n🚨 URGENT - RESTOCK NEEDED:")
            print("-" * 30)
            for item, data in urgent_items:
                print(f"{item:15} | Current: {data['current_stock']:3d} | "
                        f"Need: {data['predicted_consumption']:3d} | "
                        f"Order: {data['restock_needed']:3d}")

        if sufficient_items:
            print(f"\n✅ SUFFICIENT STOCK ({len(sufficient_items)} items):")
            print("-" * 30)
            for item, data in sufficient_items[:5]:  # Show top 5
                print(f"{item:15} | Current: {data['current_stock']:3d} | "
                        f"Need: {data['predicted_consumption']:3d}")

            if len(sufficient_items) > 5:
                print(f"... and {len(sufficient_items) - 5} more items with sufficient stock")



In [10]:
# Initialize the Restaurant Inventory Forecaster
print("🚀 Starting Restaurant Inventory Forecasting System...")

# Initialize forecaster
forecaster = RestaurantInventoryForecaster()

# Generate sample data (you can replace this with your real data)
print("📊 Generating sample restaurant data...")
df = forecaster.generate_sample_data(days=365)  # 6 months of data

print(f"✅ Generated {len(df)} records for {len(forecaster.food_items)} food items")
print(f"📅 Data range: {df['date'].min()} to {df['date'].max()}")

# Display sample of the generated data
print("\n📋 Sample of generated data:")
print(df.head(10))

🚀 Starting Restaurant Inventory Forecasting System...
📊 Generating sample restaurant data...
✅ Generated 5475 records for 15 food items
📅 Data range: 2024-09-19 02:59:52.697872 to 2025-09-18 02:59:52.697872

📋 Sample of generated data:
                        date       food_item  day_of_week  month  is_weekend  \
0 2024-09-19 02:59:52.697872  Chicken Breast            3      9           0   
1 2024-09-19 02:59:52.697872      Beef Steak            3      9           0   
2 2024-09-19 02:59:52.697872          Salmon            3      9           0   
3 2024-09-19 02:59:52.697872           Pasta            3      9           0   
4 2024-09-19 02:59:52.697872            Rice            3      9           0   
5 2024-09-19 02:59:52.697872        Tomatoes            3      9           0   
6 2024-09-19 02:59:52.697872          Onions            3      9           0   
7 2024-09-19 02:59:52.697872         Lettuce            3      9           0   
8 2024-09-19 02:59:52.697872          Cheese

In [13]:
# Train the forecasting models
print("🤖 Training forecasting models...")
forecaster.train_models(df)

# Display some statistics about the training data
print(f"\n📊 Training Data Statistics:")
print(f"Total records: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Average daily consumption by item:")
consumption_stats = df.groupby('food_item')['consumption'].agg(['mean', 'std']).round(1)
print(consumption_stats)

🤖 Training forecasting models...
✅ Trained models for 15 food items

📊 Training Data Statistics:
Total records: 5475
Date range: 2024-09-19 02:59:52.697872 to 2025-09-18 02:59:52.697872
Average daily consumption by item:
                mean   std
food_item                 
Beef Steak      16.5   5.8
Bell Peppers    10.8   3.9
Bread           40.6  12.8
Carrots          8.4   3.0
Cheese          13.1   4.5
Chicken Breast  27.9  10.8
Garlic           2.7   1.1
Lettuce         10.7   3.7
Mushrooms        7.0   2.6
Onions          16.9   6.0
Pasta           34.3  11.5
Potatoes        24.3   8.8
Rice            22.5   8.0
Salmon          12.9   4.8
Tomatoes        20.1   7.1


In [15]:
# Set current stock levels (replace with your actual inventory)
current_stock = {
    'Chicken Breast': 300, 'Beef Steak': 25, 'Salmon': 18,
    'Pasta': 120, 'Rice': 80, 'Tomatoes': 30, 'Onions': 40,
    'Lettuce': 15, 'Cheese': 35, 'Bread': 50, 'Potatoes': 60,
    'Carrots': 25, 'Bell Peppers': 20, 'Mushrooms': 15, 'Garlic': 10
}

print("📦 Current inventory levels:")
for item, stock in current_stock.items():
    print(f"{item:15}: {stock:3d} units")

# Generate 7-day forecast
print("\n🔮 Generating 7-day inventory forecast...")
forecast = forecaster.forecast_inventory_needs(
    days_ahead=7,
    current_stock=current_stock,
    safety_margin=0.2  # 20% safety margin
)

📦 Current inventory levels:
Chicken Breast : 300 units
Beef Steak     :  25 units
Salmon         :  18 units
Pasta          : 120 units
Rice           :  80 units
Tomatoes       :  30 units
Onions         :  40 units
Lettuce        :  15 units
Cheese         :  35 units
Bread          :  50 units
Potatoes       :  60 units
Carrots        :  25 units
Bell Peppers   :  20 units
Mushrooms      :  15 units
Garlic         :  10 units

🔮 Generating 7-day inventory forecast...


In [16]:
# Display the forecast results
forecaster.print_forecast_summary(forecast, days_ahead=7)

print(f"\n💡 Pro Tip: Update current_stock with your actual inventory levels!")
print("📈 This system learns from historical patterns to optimize your ordering.")

# Let's also show some detailed predictions for specific items
print(f"\n📋 Detailed daily predictions for top items:")
urgent_items = [item for item, data in forecast.items() if data['status'] == 'RESTOCK NEEDED']

if urgent_items:
    for item in urgent_items[:3]:  # Show top 3 urgent items
        print(f"\n{item}:")
        for day_pred in forecast[item]['daily_predictions']:
            print(f"  {day_pred['date']}: {day_pred['predicted_consumption']} units")
else:
    print("No urgent restocking needed!")


🍽️  RESTAURANT INVENTORY FORECAST - NEXT 7 DAYS

🚨 URGENT - RESTOCK NEEDED:
------------------------------
Beef Steak      | Current:  25 | Need: 111 | Order: 108
Salmon          | Current:  18 | Need:  90 | Order:  90
Pasta           | Current: 120 | Need: 251 | Order: 181
Rice            | Current:  80 | Need: 136 | Order:  83
Tomatoes        | Current:  30 | Need: 151 | Order: 151
Onions          | Current:  40 | Need:  95 | Order:  74
Lettuce         | Current:  15 | Need:  75 | Order:  75
Cheese          | Current:  35 | Need:  98 | Order:  82
Bread           | Current:  50 | Need: 241 | Order: 239
Potatoes        | Current:  60 | Need: 157 | Order: 128
Carrots         | Current:  25 | Need:  48 | Order:  32
Bell Peppers    | Current:  20 | Need:  73 | Order:  67
Mushrooms       | Current:  15 | Need:  51 | Order:  46
Garlic          | Current:  10 | Need:  20 | Order:  14

✅ SUFFICIENT STOCK (1 items):
------------------------------
Chicken Breast  | Current: 300 | Need: 215

💡 

In [17]:
# Test individual predictions for specific scenarios
print("🧪 Testing individual predictions:")

# Test different weather conditions
test_date = datetime.now() + timedelta(days=1)
test_item = 'Chicken Breast'

print(f"\nPredictions for {test_item} tomorrow ({test_date.strftime('%Y-%m-%d')}):")
print(f"  Sunny weather: {forecaster.predict_consumption(test_item, test_date, weather='sunny')} units")
print(f"  Rainy weather: {forecaster.predict_consumption(test_item, test_date, weather='rainy')} units")
print(f"  Special event (sunny): {forecaster.predict_consumption(test_item, test_date, weather='sunny', special_event=1)} units")

# Test weekend vs weekday
weekday_date = datetime.now() + timedelta(days=1)  # Adjust to get a weekday
while weekday_date.weekday() >= 5:  # Find next weekday
    weekday_date += timedelta(days=1)

weekend_date = weekday_date + timedelta(days=(5 - weekday_date.weekday()))  # Find next Saturday

print(f"\nWeekday vs Weekend predictions for {test_item}:")
print(f"  Weekday ({weekday_date.strftime('%A, %Y-%m-%d')}): {forecaster.predict_consumption(test_item, weekday_date)} units")
print(f"  Weekend ({weekend_date.strftime('%A, %Y-%m-%d')}): {forecaster.predict_consumption(test_item, weekend_date)} units")

🧪 Testing individual predictions:

Predictions for Chicken Breast tomorrow (2025-09-20):
  Sunny weather: 42 units
  Rainy weather: 33 units
  Special event (sunny): 45 units

Weekday vs Weekend predictions for Chicken Breast:
  Weekday (Monday, 2025-09-22): 27 units
  Weekend (Saturday, 2025-09-27): 42 units


In [18]:
# Save the dataset to CSV file
df.to_csv('restaurant_inventory_data.csv', index=False)
print("✅ Dataset saved as 'restaurant_inventory_data.csv'")

✅ Dataset saved as 'restaurant_inventory_data.csv'
